# 4  - Ereduak

## 0 - Datuak eta ereduen errendimendua

Gai hontan ikusiko ditufun ereduak frogatzeko, aurreko gaietako bi datu base erabiliko ditugu.

In [ ]:
!pip install git+https://github.com/mpenagar/ml_course.git
import joblib
import ml_course
from ml_course import SimpleRegressor, ComplexRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV

### Erregresio ataza - *California Housing Prices*:

In [ ]:
X_train, y_train, X_test, y_test = joblib.load("housing.data")

`SimpleRegressor` eredua nahiko sinplea zen, ez zuen hiperparametrorik:

In [ ]:
model = SimpleRegressor()
model.fit(X_train, y_train)
y_train_predict = model.predict(X_train)
y_test_predict = model.predict(X_test)
print(f'Train RMSE: ${root_mean_squared_error(y_train, y_train_predict):.2f}')
print(f'Test  RMSE: ${root_mean_squared_error(y_test, y_test_predict):.2f}')

`ComplexRegressor` ereduak bi hiperparametro zituen, beraz `GridSearchCV` erabiltzen genuen eredua doitzeko:

In [ ]:
grid = {
    'complexity' : [5, 10, 15, 20, 25, 30, 35] ,
    'min_split_size' : [2, 4, 8, 16, 32, 64, 128]
}
grid_search = GridSearchCV(
    estimator=ComplexRegressor(random_state=42),
    param_grid=grid,
    cv = 5,
    scoring='neg_mean_squared_error' #negative, errendimendua baita, ez galera.
)
grid_search.fit(X_train, y_train)
print(f'GridSearchCV best_params: {grid_search.best_params_}')
print(f'GridSearchCV RMSE: ${(-grid_search.best_score_)**0.5:.2f}')

`complexity=25` eta `min_split_size=64` inguruan:

In [ ]:
grid = {
    'complexity' : [21, 23, 25, 27, 29] ,
    'min_split_size' : [40, 50, 64, 80, 100]
}
grid_search = GridSearchCV(
    estimator=ComplexRegressor(random_state=42),
    param_grid=grid,
    cv = 5,
    scoring='neg_mean_squared_error' #negative, errendimendua baita, ez galera.
)
grid_search.fit(X_train, y_train)
print(f'GridSearchCV best_params: {grid_search.best_params_}')
print(f'GridSearchCV RMSE: ${(-grid_search.best_score_)**0.5:.2f}')

`$59850.16` &rarr; `$59806.94`, bilaketa jarraitu `complexity=23` eta `min_split_size=50` inguruan:

In [ ]:
grid = {
    'complexity' : [22, 23, 24] ,
    'min_split_size' : [42, 44, 46, 48, 50, 52, 54, 56, 58, 60]
}
grid_search = GridSearchCV(
    estimator=ComplexRegressor(random_state=42),
    param_grid=grid,
    cv = 5,
    scoring='neg_mean_squared_error' #negative, errendimendua baita, ez galera.
)
grid_search.fit(X_train, y_train)
print(f'GridSearchCV best_params: {grid_search.best_params_}')
print(f'GridSearchCV RMSE: ${(-grid_search.best_score_)**0.5:.2f}')

`$59806.94` &rarr; `$59636.42`, bilaketa jarraitu  `complexity=22` eta `min_split_size=58` inguruan:

In [ ]:
grid = {
    'complexity' : [20, 21, 22, 23] ,
    'min_split_size' : [57, 58, 59]
}
grid_search = GridSearchCV(
    estimator=ComplexRegressor(random_state=42),
    param_grid=grid,
    cv = 5,
    scoring='neg_mean_squared_error' #negative, errendimendua baita, ez galera.
)
grid_search.fit(X_train, y_train)
print(f'GridSearchCV best_params: {grid_search.best_params_}')
print(f'GridSearchCV RMSE: ${(-grid_search.best_score_)**0.5:.2f}')

Bilaketa amaitutzat eman dezakegu  `complexity=21` eta `min_split_size=58` hiperparametroekin.

In [ ]:
model = grid_search.best_estimator_
model.fit(X_train, y_train)
y_train_predict = model.predict(X_train)
y_test_predict = model.predict(X_test)
print(f'Train RMSE: ${root_mean_squared_error(y_train, y_train_predict):.2f}')
print(f'Test  RMSE: ${root_mean_squared_error(y_test, y_test_predict):.2f}')

### Sailkapen ataza (3 klase) - *Iris Datubasea*

## 1 - Nomenklatura, Notazioa eta Oinarrizko Formalismoa

Eredu guztietan hizkuntza matematiko berbera erabiltzeko, komenigarria da notazio konbentzio batzuk finkatzea :

**Eskalarrak.** Letra xehe etzanez adierazten dira ($x, y, b, \lambda, \alpha$). Konstanteak (lagin kopurua edo ezaugarri kopurua) letra larri etzanez adierazten dira ($N, D, K$)

**Bektoreak.** Letra xehe lodiz adierazten dira ($\mathbf{x}, \mathbf{w}, \mathbf{y}$). Konbentzioz, bektore guztiak **zutabe-bektoreak** dira ($\mathbf{x} \in \mathbb{R}^{D \times 1}$). Beraz, bi bektoreren arteko biderkadura eskalarra honela idazten da:
$$\mathbf{w}^T \mathbf{x} = \mathbf{x}^T \mathbf{w} = \sum_{j=1}^{D} w_j x_j$$

**Matrizeak.** Letra larri lodiz adierazten dira ($\mathbf{X}, \mathbf{W}$). Askotan erabiliko dugun $\mathbf{X} \in \mathbb{R}^{N \times D}$ datu-matrizeak, adibidez, laginak errepresentatzen dituzten $D$ tamainako $\mathbf{x}_i^T$ ezaugarri-bektore irauliak ditu bere $N$ errenkadetan (bektore guztiak **zutabe-bektoreak** direla gogoratu):

$$\mathbf{X} = \begin{bmatrix} \mathbf{x}_1^T \\ \mathbf{x}_2^T \\ \vdots \\ \mathbf{x}_N^T \end{bmatrix} = [\mathbf{x}_1^T; \dots; \mathbf{x}_N^T] = [\mathbf{x}_1, \dots, \mathbf{x}_N]^T \in \mathbb{R}^{N \times D}$$

Erabiliko dugun notazioan, konkatenazio horizontala (bata bestearen ondoan) $[a,b,c]$ bidez adierazten da eta konkatenazio bertikala (bata bestearen azpian) $[a;b;c]$ bidez

**Iragarpenak eta Estimazioak ($\hat{\phantom{x}}$ *txapela*).** $\hat{y}_i$ (iragarritako balioa edo klasea), $\hat{p}_i$ (iragarritako probabilitatea), $\hat{\boldsymbol{\theta}}$ (doitutako parametroak). 

**$\mathbf{1}$ bektorea.** $1$-ez osotutako bektorea da, $\mathbf{1} = [1,\dots,1]^T \in \mathbb{R}^K$. Askotan erabili ohi da beste edozein eskalarrez osotutako bektore bat adierazteko:
$$a \mathbf{1} = [a,\dots,a]^T$$

**Alborapena (Bias / Intercept) eta notazio trinkoa ($\tilde{\phantom{x}}$ *tildea*).** Eredu lineal batetan
$$y = \mathbf{w}^T \mathbf{x} + b$$
dugu, $\mathbf{w} \in \mathbb{R}^D$ pisu-bektorea delarik eta $b \in \mathbb{R}$ alborapena. Notazioa sinplifikatzeko, batzuetan $\mathbf{x}$-ri $x_0 = 1$ elementua gehitzen zaio ($\tilde{\mathbf{x}} = [1, x_1, \dots, x_D]^T$), pisu-bektorean $b$ alborapena barneratuz ($\mathbf{w} = [w_0=b, w_1, \dots, w_D]$):
$$y = \mathbf{w}^T \tilde{\mathbf{x}}$$

## 2 - Erregresio Lineala

**Erabilpena:** Erregresioa  ($\hat{y} \in \mathbb{R}$)

### Formalismoa

Helburu aldagaia sarrerako ezaugarrien konbinazio lineal gisa ereduztatzen da:

$$\hat{y}_i = f(\mathbf{x}_i; \mathbf{w}, b) = \mathbf{w}^T \mathbf{x}_i + b$$

Notazio trinkoa erabiliz:

$$\hat{y}_i = \mathbf{w}^T \tilde{\mathbf{x}}_i$$

Matrize-forman datu-multzo osorako:

$$\hat{\mathbf{y}} = \left( \mathbf{w}^T \, [\mathbf{x}_1, \dots, \mathbf{x}_N] \right)^T + b \mathbf{1} = \left( \mathbf{w}^T \mathbf{X}^T \right)^T + b \mathbf{1} = \mathbf{X}\mathbf{w} + b \mathbf{1}$$

Matrize-forman datu-multzo osorako eta notazio trinkoa erabiliz:

$$\hat{\mathbf{y}} = \tilde{\mathbf{X}} \mathbf{w}$$


### Errepresentazio grafikoa

<img src="img/erregresio-lineala-1d.jpg" style="display: block; margin: 0 auto; width: 60%; height: auto;" alt="erregresio lineala 1D"/>

<img src="img/erregresio-lineala-2d.png" style="display: block; margin: 0 auto; width: 60%; height: auto;" alt="erregresio lineala 2D"/>

### Galera-funtzioa

**Karratu Txikienen Metodo Arrunta** (*OLS, Ordinary Least Squares*) erabiltzen da, zeinetan Errore Koadratikoaren Batezbestekoa (*MSE*) izango den galera funtzioa: 

$$\mathcal{L}(\mathbf{y}, \hat{\mathbf{y}}) = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2 = \frac{1}{N} \sum_{i=1}^{N} (y_i - (\mathbf{w}^T \mathbf{x}_i + b))^2 = \frac{1}{N} \Vert{}\mathbf{y} - (\mathbf{X}\mathbf{w} + b\mathbf{1})\Vert{}_2^2$$

Notazio trinkoa erabiliz:

$$\mathcal{L}(\mathbf{y}, \hat{\mathbf{y}}) = \frac{1}{N} \Vert{}\mathbf{y} - \tilde{\mathbf{X}} \mathbf{w}\Vert{}_2^2$$

### Doitzea

Beste eredu askotan bezala gradientearen jaitsieraz ebatz badaiteke ere, erregresio linealak ebazpen analitiko itxia du:

$$\hat{\mathbf{w}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}_{train}$$

**OHARRA:** Ezaugarrien artean kolinealtasun handia dagoenean (zutabe bat beste baten konbinazio lineala edo ia lineala denean) $\mathbf{X}^T\mathbf{X}$ matrizea singularra edo ia singularra (*ill-conditioned*) bihurtzen da. Bere determinantea zerora hurbiltzen da, eta autobalio (eigenvalues) batzuk $\approx 0$ dira. Ondorioz, $(\mathbf{X}^T\mathbf{X})^{-1}$ alderantzizko matrizea kalkulatzean, balioak infiniturantz jotzen dute eta $\hat{\mathbf{w}}$ pixuen bariantza izugarri handitzen da (datuetan aldaketa txiki batek sekulako aldaketa eragiten du pisuetan).

### Erregularizazioak

Erregularizazioaren helburua gaindoikuntza saihestea da. Orohar, erregularizazioa ereduaren $\boldsymbol{\theta}$ barne parametroek (*pixuek*) har ditzazketen balioen magnitudeen gaineko $\Omega(\boldsymbol{\theta})$ zigor bat izan ohi da. Zigor edo erregularizazio-termino hau $\lambda$ faktore batez eskalatu eta $\mathcal{L}(\mathbf{y}, \hat{\mathbf{y}})$ galera funtzioari gehitu ohi zaio, benetan minimizatuko den $\mathcal{J}(\mathbf{y},\boldsymbol{\theta})$ **Helburu-funtzioa** sortuz:

$$\underbrace{\mathcal{J}(\mathbf{y},\boldsymbol{\theta})}_{\substack{\text{Helburu-} \\ \text{funtzioa}}} = \underbrace{\mathcal{L}(\mathbf{y}, \hat{\mathbf{y}})}_{\substack{\text{Galera-} \\ \text{funtzioa}}} + \lambda \cdot \underbrace{\Omega(\boldsymbol{\theta})}_{\substack{\text{Erregularizazio-} \\ \text{terminoa}}}$$

Erregresio linealaren kasuan, $\boldsymbol{\theta} = \mathbf{w}$.

**OHARRA:** Orohar, ereduetako alborapen terminoek ez dute erregularizazioetan parte hartzen. Erregresio linealaren kasuan, notazio trinkoa erabiltzen badugu ere, erregularizazioei dagokienez,

$$\mathbf{w} = [w_1, \dots, w_D]$$

**1 - Ridge Erregularizazioa ($L_2$ penalizazioa - Ridge)**. Erregularizazio terminoa pisu-bektorearen $L_2$ norma da (norma euklidearraren karratua):

$$\Omega(\mathbf{w}) = \Vert{}\mathbf{w}\Vert{}_2^2$$

Helburu-funtzioa:

$$\mathcal{J}(\mathbf{y},\mathbf{w}) = \text{MSE} + \lambda \Vert{}\mathbf{w}\Vert{}_2^2 = \text{MSE} + \lambda \sum_{j=1}^{D} w_j^2$$

Ridge erregresioaren ezaugarriak:
* Pisu guztiak zerorantz uzkurtzen ditu modu proportzionalean, baina ez ditu guztiz $0$ bihurtzen.
* Kolinaltasun-arazoak egonkortzen ditu, $\lambda \mathbf{I}$ terminoa gehitzen diolako ebazpen analitiko itxiko $\mathbf{X}^T\mathbf{X}$ matrizearen diagonal nagusiari:
$$\hat{\mathbf{w}}_{\text{Ridge}} = (\mathbf{X}^T\mathbf{X} + \lambda \mathbf{I})^{-1}\mathbf{X}^T\mathbf{y}$$

**2 - LASSO Erregularizazioa ($L_1$ penalizazioa - Lasso):**. Erregularizazio terminoa pisu-bektorearen $L_1$ norma da (*Manhattan* distantzia):

$$\Omega(\mathbf{w}) = \Vert{}\mathbf{w}\Vert{}_1$$

Helburu-funtzioa:

$$\mathcal{J}(\mathbf{y},\mathbf{w}) = \text{MSE} + \lambda \Vert{}\mathbf{w}\Vert{}_1 = \text{MSE} + \lambda \sum_{j=1}^{D} \vert{}w_j\vert{}$$

LASSO erregresioaren ezaugarriak:
* Garrantzi gutxiko ezaugarrien pisuak $0$ bihurtzen ditu ($w_j = 0$).
* Ezaugarrien hautaketa automatikoa (feature selection) egiten du (eredu sakabanatuak edo sparse).

**3 - ElasticNet Erregularizazioa**.  $0 \le r \le 1$ parametro baten araberako $L_1$ eta $L_2$ penalizazioen konbinazioa da:

$$\Omega(\mathbf{w}) = r \Vert{}\mathbf{w}\Vert{}_1 + \frac{1 - r}{2} \Vert{}\mathbf{w}\Vert{}_2^2$$

$$  $$

### Ahultasunak

* Hipotesi zurrunegia (Alborapen handia): Erlazioa lineala dela suposatzen du; ezin ditu berez kurbadurak edo interakzioak harrapatu (polinomioak eskuz sortu ezean).
* Muturreko balioekiko (Outliers) sentikortasuna: Errorea berretzen duenez ($(y_i - \hat{y}_i)^2$), outlier bakar batek malda osoa desbideratu dezake.
* Eskalarekiko menpekotasuna (Ridge/LASSO-n): Ezaugarriak estandarizatuta ez badaude (StandardScaler), eskala txikiko aldagai batek $w_j$ handia beharko du eta erregularizazioak bidegabeki zigortuko du.
* Multikolinealtasuna: Bi ezaugarri oso korrelazionatuta badaude, OLS-n pisuak ezegonkor bihurtzen dira ($\mathbf{X}^T\mathbf{X}$ ia singularra da).

### Kodea

*Scikit-learn*ek erregresio lineala egiteko oinarrizko bi eredu eskaintzen ditu `sklearn.linear_model` moduluan:

* [`Ridge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) - *Linear least squares with l2 regularization*
* [`Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) - *Linear Model trained with L1 prior as regularizer (aka the Lasso)*

Eredu bakoitzak parametro asko ditu, hauek dira nabarmenenak:
* **`Ridge(alpha=1.0, max_iter=None, tol=0.0001, solver='auto', random_state=None)`**:
   * `alpha [1.0]`: $L_2$ erregularizazioaren indarra kontrolatzen duen hiperparametro nagusia da (teorian $\lambda$ bezala idatzi duguna). Zenbat eta balio handiagoa izan, orduan eta gehiago murriztuko dira pisuak gaindoikuntza (overfitting) saihesteko; 0 balioarekin, berriz, erregresio lineal arrunt (OLS) bihurtzen da.
   * `solver ['auto']`: Optimizazio-problema ebazteko erabiliko den barne-algoritmoa aukeratzen du (adibidez, 'cholesky' soluzio analitiko zuzenerako, edo 'sag' / 'saga' datu-multzo handietan prozedura iteratiboak erabiltzeko). 'auto' utzita, Scikit-learnek datuen tamainaren eta motaren arabera egokiena aukeratzen du automatikoki.
   * `max_iter [None]` eta `tol [0.0001]`: Ebatzaile (solver) iteratiboetan soilik eragiten dute, algoritmoak konbergentziara iristeko gehienez zenbat iterazio egingo dituen eta gelditzeko tolerantzia-atalasea zein den finkatuz.
   * `random_state [None]`: Ausazko zenbaki-sortzailearen hazia (seed) finkatzen du, emaitzen erreproduzigarritasuna bermatzeko. Soilik du eragina datuak ausaz nahasten dituzten ebatzaile estokastikoak erabiltzean (solver='sag' edo solver='saga').

* **`Lasso(alpha=1.0, fit_intercept=True, precompute=False, copy_X=True, max_iter=1000, tol=0.0001, warm_start=False, positive=False, random_state=None, selection='cyclic')`**:
   * `alpha [1.0]`: $L_1$ erregularizazioaren indarra kontrolatzen duen hiperparametroa da (teorian $\lambda$). Balioa handitzean, garrantzi gutxiko aldagaien pisuak ($\mathbf{w}_j$) erabat zero bihurtzen ditu, aldagaien aukeraketa automatikoa (feature selection) eginez.
   * `max_iter [1000]` eta `tol [0.0001]`: Koordenatuen jaitsiera algoritmoak konbergentziara iristeko gehienez zenbat iterazio egingo dituen (hemen 1000 da zuzenean, beti iteratiboa delako) eta gelditzeko tolerantzia-atalasea zein den finkatzen dute.
   * `selection ['cyclic']`: Iterazio bakoitzean pisuak zein ordenatan eguneratzen diren zehazten du: 'cyclic' aukerak aldagaiak banan-banan ordena finkoan eguneratzen ditu, eta 'random' aukerak ausazko koordenatu bat hautatzen du urrats bakoitzean (askotan konbergentzia azkartuz, batez ere tolerantzia handia denean).
   * `random_state [None]`: Ausazko zenbaki-sortzailearen hazia (seed) finkatzen du emaitzen erreproduzigarritasuna bermatzeko; selection='random' aukeratzen denean erabiltzen da eguneratuko den hurrengo pisua ausaz aukeratzeko.
OHARRA: Lasso ereduak ez du solver parametrorik, barnean beti algoritmo bakarra erabiltzen duelako: koordenatuen jaitsiera (*Coordinate Descent*).

*Scikit-learn*ek erregresio linealerako eskaintzen duen eredu sorta oso handia da, hemen soilik bi aztertu ditugu. `sklearn.linear_model` moduluan guzti hauek dituzu eskuragai:

* Oinarrizko ereduak eta Erregularizatuak: LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor
* Dimentsio handietarako eta aukeraketa-bideetarako (Path algorithms): Lars, LassoLars, OrthogonalMatchingPursuit
* Eredu Bayestarrak: BayesianRidge, ARDRegression
* Muturreko balioekiko eredu sendoak (Outliers): HuberRegressor, RANSACRegressor, TheilSenRegressor, QuantileRegressor
* Eredu Lineal Orokortuak (GLM): PoissonRegressor, GammaRegressor, TweedieRegressor
* Ataza-anitzeko ereduak (Multi-task): MultiTaskLasso, MultiTaskElasticNet, MultiTaskLassoCV, MultiTaskElasticNetCV
* Hiperparametroen doikuntza automatikodunak: RidgeCV, LassoCV, ElasticNetCV, LarsCV, LassoLarsCV, OrthogonalMatchingPursuitCV, LassoLarsIC

## 3 - Erregresio Logistikoa (*Logistic Regression*)

## 4 - Euskarri-Bektore Makinak (*SVM - Support Vector Machines*)

## 5 - Erabaki-Zuhaitzak (*Decision Trees*)

## 6 - Ausazko Basoak (*Random Forests*)